# Autoshop-Microservices in Backstage anzeigen

Minimaler Ablauf für Microservices, die bereits in Kubernetes laufen. Backstage erhält Leserechte, die Kubernetes-Workloads werden beschriftet und als Catalog-Entities registriert.

## 1. Umgebung und Microservices festlegen

Passe Namespace, Image und die Namen der vorhandenen Kubernetes Deployments an. Der Schlüssel wird gleichzeitig als Name der Backstage-Komponente und als Kubernetes-ID verwendet.

In [ ]:
import os

os.environ['BACKSTAGE_NAMESPACE'] = 'backstage'
os.environ['AUTOSHOP_NAMESPACE'] = 'autoshop'
os.environ['BACKSTAGE_IMAGE'] = 'registry.example.com/mybackstage:1.1.0'

# Backstage-Komponente: vorhandenes Kubernetes Deployment
services = {
    'autoshop-frontend': 'autoshop-frontend',
    'autoshop-orders': 'autoshop-orders',
    'autoshop-inventory': 'autoshop-inventory',
    'autoshop-payments': 'autoshop-payments',
}

services

## 2. Kubernetes-Leserechte für Backstage erstellen

Backstage verwendet einen eigenen ServiceAccount. Die Rolle erlaubt ausschliesslich das Lesen der Ressourcen, die im Kubernetes-Tab angezeigt werden.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: ServiceAccount
metadata:
  name: backstage
  namespace: ${BACKSTAGE_NAMESPACE}
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: backstage-kubernetes-reader
rules:
  - apiGroups: ['']
    resources: ['pods', 'pods/log', 'services', 'configmaps', 'events']
    verbs: ['get', 'list', 'watch']
  - apiGroups: ['apps']
    resources: ['deployments', 'replicasets', 'statefulsets', 'daemonsets']
    verbs: ['get', 'list', 'watch']
  - apiGroups: ['batch']
    resources: ['jobs', 'cronjobs']
    verbs: ['get', 'list', 'watch']
  - apiGroups: ['autoscaling']
    resources: ['horizontalpodautoscalers']
    verbs: ['get', 'list', 'watch']
  - apiGroups: ['networking.k8s.io']
    resources: ['ingresses']
    verbs: ['get', 'list', 'watch']
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: backstage-kubernetes-reader
subjects:
  - kind: ServiceAccount
    name: backstage
    namespace: ${BACKSTAGE_NAMESPACE}
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: backstage-kubernetes-reader
EOF

## 3. ServiceAccount dem Backstage Deployment zuweisen

Damit Backstage innerhalb des Clusters auf die Kubernetes API zugreifen kann, wird der neue ServiceAccount im bestehenden Deployment gesetzt.

In [ ]:
%%bash
kubectl patch deployment backstage \
  --namespace "${BACKSTAGE_NAMESPACE}" \
  --type merge \
  --patch '{"spec":{"template":{"spec":{"serviceAccountName":"backstage"}}}}'

## 4. Kubernetes-Plugin installieren

Das Frontend-Plugin erzeugt den Kubernetes-Tab. Das Backend-Plugin liest die Ressourcen über die Kubernetes API.

In [ ]:
%%bash
cd ~/mybackstage

yarn --cwd packages/app add @backstage/plugin-kubernetes
yarn --cwd packages/backend add @backstage/plugin-kubernetes-backend

## 5. Backend-Plugin aktivieren

Beim neuen Backstage-Backend wird das Kubernetes-Plugin in `packages/backend/src/index.ts` registriert. Die Zelle fügt die Zeile nur ein, wenn sie noch fehlt.

In [ ]:
from pathlib import Path

backend_index = Path.home() / 'mybackstage/packages/backend/src/index.ts'
text = backend_index.read_text(encoding='utf-8')
plugin_line = "backend.add(import('@backstage/plugin-kubernetes-backend'));"

if plugin_line not in text:
    text = text.replace('backend.start();', f"{plugin_line}\n\nbackend.start();")
    backend_index.write_text(text, encoding='utf-8')

print(plugin_line)

## 6. Kubernetes-Konfiguration für Backstage erstellen

Backstage läuft im gleichen Cluster und verwendet deshalb den zugewiesenen ServiceAccount. Zusätzlich wird der Kubernetes-Tab auf den Component-Seiten aktiviert.

In [ ]:
%%bash
cd ~/mybackstage

cat > app-config.kubernetes.yaml <<'EOF'
app:
  extensions:
    - entity-content:kubernetes/kubernetes

kubernetes:
  serviceLocatorMethod:
    type: multiTenant
  clusterLocatorMethods:
    - type: config
      clusters:
        - name: local-kubernetes
          authProvider: serviceAccount
          skipMetricsLookup: true

catalog:
  locations:
    - type: file
      target: ../../examples/autoshop-components.yaml
EOF

## 7. Kubernetes-Workloads beschriften

Das Label `backstage.io/kubernetes-id` verbindet eine Backstage-Komponente mit ihrem Deployment, den Pods und dem Service.

In [ ]:
import os
import subprocess

namespace = os.environ['AUTOSHOP_NAMESPACE']

for component, deployment in services.items():
    subprocess.run([
        'kubectl', 'label', 'deployment', deployment,
        '--namespace', namespace,
        f'backstage.io/kubernetes-id={component}',
        '--overwrite'
    ], check=True)

    # Das Pod-Template wird ebenfalls beschriftet, damit neu erzeugte Pods gefunden werden.
    subprocess.run([
        'kubectl', 'patch', 'deployment', deployment,
        '--namespace', namespace,
        '--type', 'merge',
        '--patch', '{"spec":{"template":{"metadata":{"labels":{"backstage.io/kubernetes-id":"' + component + '"}}}}}'
    ], check=True)

    # Ein gleichnamiger Service wird beschriftet, falls er existiert.
    subprocess.run([
        'kubectl', 'label', 'service', deployment,
        '--namespace', namespace,
        f'backstage.io/kubernetes-id={component}',
        '--overwrite'
    ], check=False)

## 8. Backstage Catalog-Entities erzeugen

Für jeden Microservice wird eine Component-Entity erstellt. Die Annotationen enthalten dieselbe Kubernetes-ID und den Namespace der laufenden Workloads.

In [ ]:
from pathlib import Path
import os

namespace = os.environ['AUTOSHOP_NAMESPACE']
documents = []

for component in services:
    documents.append(f'''apiVersion: backstage.io/v1alpha1
kind: Component
metadata:
  name: {component}
  namespace: default
  title: {component.replace('-', ' ').title()}
  annotations:
    backstage.io/kubernetes-id: {component}
    backstage.io/kubernetes-namespace: {namespace}
  tags:
    - autoshop
    - kubernetes
spec:
  type: service
  lifecycle: production
  owner: user:default/guest
  system: autoshop
''')

target = Path.home() / 'mybackstage/examples/autoshop-components.yaml'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('---\n'.join(documents), encoding='utf-8')
print(target.read_text(encoding='utf-8'))

## 9. Backstage neu bauen und Image veröffentlichen

Die Plugins, Konfiguration und Catalog-Datei werden in ein neues Backstage-Image eingebaut. Melde dich vorher bei deiner Container Registry an.

In [ ]:
%%bash
cd ~/mybackstage

yarn install --immutable
yarn tsc
yarn build:backend

docker image build \
  --file packages/backend/Dockerfile \
  --tag "${BACKSTAGE_IMAGE}" \
  .

docker push "${BACKSTAGE_IMAGE}"

## 10. Neues Backstage-Image starten

Das bestehende Deployment wird auf das neue Image aktualisiert. Danach erscheinen die Microservices im Catalog und auf jeder Component-Seite im Tab **Kubernetes**.

In [ ]:
%%bash
kubectl set image deployment/backstage \
  --namespace "${BACKSTAGE_NAMESPACE}" \
  backstage="${BACKSTAGE_IMAGE}"

kubectl rollout status deployment/backstage \
  --namespace "${BACKSTAGE_NAMESPACE}"